In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PostgreSQL Autodownload Example") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/03 21:53:09 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/11/03 21:53:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/tem/de_lrn_prcts/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/tem/.ivy2.5.2/cache
The jars for the packages stored in: /home/tem/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b64f9a14-85b6-474c-adab-a3caafd51dba;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.2 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 209ms :: artifacts dl 7ms
	:: modules in use:
	org.checkerframework#checker-qual

In [3]:
import pandas as pd

pd_orders = pd.read_csv("orders.csv").sample(n=1000)

In [4]:
schema = """
    order_id INT,
    customer_id INT,
    order_date STRING,
    product_name STRING,
    quantity INT,
    price FLOAT,
    total_amount FLOAT,
    status STRING
"""

df_orders = spark.createDataFrame(
    pd_orders,
    schema=schema
)
df_orders.printSchema()

/home/tem/de_lrn_prcts/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/tem/de_lrn_prcts/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/tem/de_lrn_prcts/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enable

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- status: string (nullable = true)



In [5]:
db_properties = {
    "user": "postgres",
    "password": "mysecretpassword",
    "driver": "org.postgresql.Driver"
}

jdbc_url = "jdbc:postgresql://172.20.131.26:5432/postgres"

df_orders.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .mode("overwrite") \
    .save()

In [6]:
df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", "orders") \
        .options(**db_properties) \
        .load()


df.show()

+--------+-----------+-------------------+-------------------+--------+-------+------------+----------+
|order_id|customer_id|         order_date|       product_name|quantity|  price|total_amount|    status|
+--------+-----------+-------------------+-------------------+--------+-------+------------+----------+
|  104412|       2028|2026-06-14 22:38:00|        Smart Watch|       3| 249.99|      749.97| Cancelled|
|  104731|       1483|2026-04-18 13:25:00|             Laptop|       2|1199.99|     2399.98| Completed|
|  101023|       1870|2026-07-07 02:08:00|      Charger Cable|       1|  19.99|       19.99| Completed|
|  104249|       1787|2026-02-12 14:36:00|             Tablet|       3| 449.99|     1349.97| Completed|
|  104145|       2269|2026-10-19 21:38:00|            Monitor|       2| 299.99|      599.98| Completed|
|  100543|       2416|2026-01-21 00:46:00|             Laptop|       1|1199.99|     1199.99| Completed|
|  100965|       1755|2026-06-22 00:27:00|Mechanical Keyboard|  

In [7]:
df_orders.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()

In [8]:
df_orders.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .option("batchsize", 1000) \
    .mode("ignore") \
    .save()

In [10]:
df_orders.coalesce(1) \
    .write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()

In [11]:
df_orders.repartition(2) \
    .write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()